# Custom Vulnerability Assessment Data Generation
This notebook illustrates how to use the Cal-Adapt vulnerability assessment support tool to build custom data metrics using climate data in the Analytics Engine. 

To execute a given 'cell' of this notebook, place the cursor in the cell and press the 'play' icon, or simply press shift+enter together. Some cells will take longer to run, and you will see a [$\ast$] to the left of the cell while AE is still working.

**Intended Application**: As a user, I want to **<span style="color:#FF0000">access climate projections data for my vulnerability assessment report</span>** by:
1. Retrieve data metrics required for planning needs

**Runtime**: With the default settings, this notebook takes approximately **several hours** to run from start to finish, depending on the metric choice. Modifications to selections may increase the runtime. 

### Step 0: Set-up

First, we'll import the Python library [climakitae](https://github.com/cal-adapt/climakitae), our AE toolkit for climate data analysis, along with this specific functions from that library that we'll use in this notebook, as well as any other necessary Python libraries to aid in analysis.

In [1]:
from climakitae.explore.vulnerability import cava_data

### Step 1: Import locations

To import your own custom locations, we recommend putting your csv file in the same folder as this notebook for ease:
1. Drag and drop a csv file into the file tree on the left hand side; or
2. Use the `upload` button (the "up arrow" symbol next to the large blue plus symbol above the file tree). 

<span style="color:#FF0000">**Formatting note**</span>: For the code cells below to work, there must be **2 columns labeled `lat` and `lon`**. Functionality to accept different labeling is forthcoming!

In the cell below, we read the csv file in. We use the HadISD station list as an example here -- you may want to replace with your own locations file!

In [2]:
# Read in dummy locations from `stations_csv` file
from climakitae.core.paths import HADISD_STATIONS_URL
from climakitae.util.utils import read_csv_file
import pandas as pd
example_locs = pd.read_csv(HADISD_STATIONS_URL, index_col=0)[['LAT_Y', 'LON_X']].rename(columns={'LAT_Y': 'lat', 'LON_X': 'lon'})

### Step 2: Retrieve metric data

The `cava_data` funciontality is designed to provide flexibility over customizable metric calculation. There are 4 customizable metrics that can be built with this functionality:
1. Likely seasonal event occurence (e.g., "likely summer night low temperature")
2. 1-in-X temperature events (e.g., "1-in-10 year maximum temperature")
3. High/Extreme Heat Index events (e.g., "how many days per year does the Heat Index exceed 90°F")
4. 1-in-X precipitation events (e.g., "1-in-100 year, 24 hour precipitation")

Below is a table outlining all avaialble arguments to the `cava_data` function. The "Required" flag notes whether the argument must be passed to start generating data. Input options for each argument is provided, as well as whether a setting is required for any of the required selections. We provide multiple examples of working with the `cava_data` function with multiple configurations.

| Argument | Options | Argument required for | Notes |
|----------|---------|-----------------------|------|
|input_locations | Pass a location via csv. | All | Option to run either a single location, or multiple when **batch_mode=True**.|
|variable | "Air Temperature at 2m", "NOAA Heat Index", "Precipitation (total)"| All | |
|approach | "Time", "Warming Level" | All | |
|downscaling_method | "Dynamical", "Statistical"| All | |
|time_start_year | Numerical (min is 1981) | Required for **approach=Time**.| |
|time_end_year | Numerical (max is 2100) | Required for **approach=Time**.| |
|historical_data| "Historical Climate", "Historical Reconstruction" | Required for **approach=Time**| **Historical Climate** ranges from 1980-2015 for WRF and 1950-2015 for LOCA2-Hybrid. **Historical Reconstruction** ranges from 1950-2022. Historical Reconstruction data cannot be combined with SSP data.|
|ssp_data | "[SSP 2-4.5]", "[SSP 3-7.0]", "[SSP 5-8.5]" | Required for **approach=Time** | Dynamical only has SSP 3-7.0, Statisical has all 3 SSP options.|
|warming_level| 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0 | Required for **approach=Warming Level**| Historical/Current period GWLs: 0.8, 1.0, 1.2. Future GWLs: 1.5, 2.0, 2.5, 3.0. |
|metric_calc| "max", "min" | Required for 1-in-X events, Heat Index, likely seasonal event | |
|heat_idx_threshold | Numerical | Required for Heat Index | Heat Index can only be calculated with **downscaling_method="Dynamical"**.|
|one_in_x | List or Numerical values | Required for 1-in-X events| Example: one_in_x = 10 *or* one_in_x = [10, 100]|
|duration | (int, "hour") | Optional for 1-in-X events | Duration of the sub-daily event window, e.g. `(3, "hour")` for a 3-hour event. Only supported for hourly WRF data. Default is unset (uses daily data).|
|groupby | (int, "day") | Optional for 1-in-X events | Groups sub-daily values into daily blocks before computing block maxima, e.g. `(1, "day")`. Only "day" units are supported. Default is `(1, "day")`.|
|grouped_duration | (int, "day") | Optional for 1-in-X events | Number of consecutive days in a multi-day event window, e.g. `(5, "day")` for a 5-day event. Requires **groupby** to be set. Only "day" units are supported. Default is unset.|
|distr| "gev", "genpareto", "gumbel", "wibull", "pearson3", "gamma"| Optional for 1-in-X events |Default set to "gev".|
|percentile | Numerical (0-100) | Required for likely seasonal event |   |
|season| "summer", "winter", "all" | Required for likely seasonal event| Default set to "all".|
|units | Temp/Heat Index: "degF", "degC", "K". Precip: "mm", "inches" | Optional | Default for temp/Heat Index is DegF. Default for precip is mm.|
|wrf_bias_adjust| True, False | Optional| Option to return only the 5 bias-adjusted WRF models. Only applicable for **downscaling_method="Dynamical"**. Default set to True.|
|export_method| "raw", "calculate", "both" | Optional | Default set to "both".|
|file_format | "NetCDF", "csv" | Optional | Default set to "NetCDF".|
|file_name | str | Optional | Custom base name for output files, no extension (e.g., `"my_output"` not `"my_output.nc"`). If None, an automatic name is generated based on the selected parameters.|
|separate_files | True, False | Optional | Option to export separate files if multiple points are passed. Default set to True.|
|batch_mode | True, False | Optional, but recommmended for multiple locations. | Option to efficiently run multiple points. Separate files for export is turned off in batch mode. Default set to False.|

The following cells illustrate several examples of how to retrieve and calculate various configurations of the `cava_data` function. Below is a list of the examples; and more are coming soon as more functionality is built in:
1. Likely seasonal event, single location, time approach, all WRF data, with custom percentile
2. Likely seasonal event, batch mode for multiple locations, time approach, all WRF data, with custom percentile
3. 1-in-X temperature event, single location, bias-adjusted WRF data only, time approach, with multiple return periods, event frequency, and distribution
4. Heat index event, single location, time approach, with custom threshold
5. Likely seasonal event, single location, warming level approach, all WRF data, with custom percentile
6. Likely seasonal event, single location, warming level approach, all LOCA2 data, with custom percentile
7. Heat index event, batch mode for multiple locations, time approach, Historical Reconstruction data, with custom threshold
8. Likely seasonal event, batch mode for multiple locations, warming levels approach, all LOCA2 data, with custom percentile
9. 1-in-X precipitation event, single location, all LOCA2 data, time approach, with custom return period
10. 1-in-X precipitation event, single location, all WRF data, time approach, with `duration`, `groupby`, and `grouped_duration`
11. Example of reading the calculated metric data via xarray for easy viewing within this notebook.

#### Example: Likely seasonal event
Example scenario: I want to calculate "likely summer day high temperature for 2030-2050, where likely is the 75th percentile, in Celsius, using all available WRF data (bias-adjusted and non-bias-adjusted), and export only the calculated metric data" for a **single location**. I would input:

In [3]:
data = cava_data(
    ## Set-up
    example_locs[:1], # select a single location        
    approach="Time",  
    time_start_year=2030, 
    time_end_year=2050,
    downscaling_method="Dynamical",  # WRF data
    wrf_bias_adjust=False, # return all WRF models
    
    ## Likely seaonal event specific arguments
    variable="Air Temperature at 2m", 
    metric_calc="max", # daily high temperature
    percentile=75, # likeliness percentile
    season="summer", # season
    units="degC", # change units
    
    ## Export
    export_method="calculate",  # export only calculated metric data
    file_format="NetCDF",
    file_name="likely_seasonal_summer_max_2030_2050",  # custom output filename (omit to use auto-generated name)
    batch_mode=True,
)

Calculating 75th percentile of Daily Max of Air Temperature at 2m for summer seasons from 2030_to_2050

--- Selecting Data Points --- 

[WARNING] Batch mode is active, 'separate_files' is being set to False. 
[WARNING] This means that your output will be a single file with all locations included. 

Batch retrieving all 1 points passed in...



/home/jovyan/.local/lib/python3.12/site-packages/climakitae/tools/batch.py:42: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!! Returned data array is huge. Operations could take 10x to infinity longer than 1GB of data !!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Processing 1 coordinate pair(s)...
  Spatial dimensions: y, x
  Data grid size: 472 x 223
  Pre-clipping to bounding box: 11 x 11 (0.1% of original)
  Found 1 valid point(s) within data extent
  Checking landmask for valid land points...
  1/1 points are on land
  Extracting gridcells...
  Point 1: (35.4342, -119.0552) -> (1015911.7307, -4089113.6619)
Done! Extracted 1 gridcell(s)

--- Loading Data into Memory ---

Processing data to read 5.61 MB of data into memory... 
[########################################] | 100% Completed | 47.20 s
Complete!

--- Calculating Metrics ---
Calculating...


Complete!

--- Exporting Metric Data ---
Exporting specified data to NetCDF...
Saving file locally as

#### Example: Likely seasonal event, in batch mode for multiple locations
Example scenario: I want to calculate "likely summer day high temperature for 2030-2050, where likely is the 75th percentile, in Celsius, using all available WRF data (bias-adjusted and non-bias-adjusted) for many locations, and export only the calculated metric data" for **multiple locations**. I would input:

In [4]:
data = cava_data(
    ## Set-up
    example_locs, # no subsetting for a single location from input list
    time_start_year=2020, 
    time_end_year=2050,
    downscaling_method="Dynamical",  # WRF data
    approach="Time",  
    wrf_bias_adjust=False, # return all WRF models
    
    ## Likely seaonal event specific arguments
    variable="Air Temperature at 2m", 
    metric_calc="max", # daily high temperature
    percentile=75, # likeliness percentile
    season="summer", # season
    units="degC", # change units
    
    ## Export
    export_method="calculate",  # export only calculated metric data
    file_format="NetCDF",
    batch_mode=True, # batch mode - optimized for multiple locations
)

Calculating 75th percentile of Daily Max of Air Temperature at 2m for summer seasons from 2020_to_2050

--- Selecting Data Points --- 

[WARNING] Batch mode is active, 'separate_files' is being set to False. 
[WARNING] This means that your output will be a single file with all locations included. 

Batch retrieving all 33 points passed in...



/home/jovyan/.local/lib/python3.12/site-packages/climakitae/tools/batch.py:42: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!! Returned data array is huge. Operations could take 10x to infinity longer than 1GB of data !!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Processing 33 coordinate pair(s)...
  Spatial dimensions: y, x
  Data grid size: 472 x 223
  Found 33 valid point(s) within data extent
  Checking landmask for valid land points...
  31/33 points are on land
  2 point(s) need neighbor search...
  Fixed 2/2 points with valid neighbors
  Extracting gridcells...
Done! Extracted 33 gridcell(s)

--- Loading Data into Memory ---

Processing data to read 273.48 MB of data into memory... 
[########################################] | 100% Completed | 16m 32s
Complete!

--- Calculating Metrics ---
Calculating...


Complete!

--- Exporting Metric Data ---
Exporting specified data to NetCDF...
Saving file locally as NetCDF4...
[###########################

#### Example: 1-in-X temperature event with multiple return periods
Example scenario: I want to calculate "1-in-10 **and** 1-in-100 year maximum temperature using the GEV distribution, in Fahrenheit, for 2070-2090, using only the bias-adjusted WRF data, and export both the raw and calculated metric data." I would input:

In [5]:
data = cava_data(
    ## Set-up
    example_locs.iloc[:1], # select a single location
    time_start_year=2070,
    time_end_year=2090,
    downscaling_method="Dynamical",  # WRF data 
    approach="Time",  
    wrf_bias_adjust=True, # return bias adjusted WRF models
    
    ## 1-in-X event specific arguments
    variable="Air Temperature at 2m",
    metric_calc="max", # daily maximum temperature
    one_in_x=[10, 100], # multiple return periods
    distr="gev", # change distribution
    units="degF", # change units
    
    ## Export
    export_method="both",
    file_format="NetCDF",
)

Calculating 1-in-10, 1-in-100 year, groupby=(1, 'day'), Max Air Temperature at 2m for all seasons from 2070_to_2090

--- Selecting Data Points --- 

Selecting data for (35.43424, -119.05524)


/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:1227: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()


Filtering WRF data for bias-adjusted simulations.

--- Loading Data into Memory ---

Point (35.445045471191406, -119.06060791015625)
Processing data to read 3.51 MB of data into memory... 
[########################################] | 100% Completed | 25.02 s
Complete!

--- Exporting Raw Data ---
Exporting specified data to NetCDF...
Saving file locally as NetCDF4...
[########################################] | 100% Completed | 101.45 ms
Saved! You can find your file in the panel to the left and download to your local machine from there.

--- Calculating Metrics ---
Calculating...

Goodness of fit for 1-in-10, 1-in-100, Air Temperature at 2m events: 
Running on simulation: WRF_MIROC6_r1i1p1f1
Found block_size of 1 in BMS attributes
The simulation WRF_MIROC6_r1i1p1f1 fitted with a gev distribution has a p-value of 0.983.

Running on simulation: WRF_EC-Earth3-Veg_r1i1p1f1
Found block_size of 1 in BMS attributes
The simulation WRF_EC-Earth3-Veg_r1i1p1f1 fitted with a gev distribution has a

/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:460: FutureWarning: In a future version of xarray the default value for compat will change from compat='equals' to compat='override'. This change will result in the following ValueError: Cannot specify both coords='different' and compat='override'. The recommendation is to set compat explicitly for this case.
  ret_vals = xr.concat(return_vals, dim="simulation", coords="different")
/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:461: FutureWarning: In a future version of xarray the default value for compat will change from compat='equals' to compat='override'. This change will result in the following ValueError: Cannot specify both coords='different' and compat='override'. The recommendation is to set compat explicitly for this case.
  p_vals = xr.concat(p_vals, dim="simulation", coords="different")


#### Example: Heat Index
Example scenario: I want to calculate "the number of days per year that the Heat Index exceeds 90°F between 2030-2060, and export only the raw data". I would input:

In [6]:
data = cava_data(
    ## Set-up
    example_locs.iloc[:1], # select a single location
    time_start_year=2030,
    time_end_year=2060,
    downscaling_method="Dynamical",  # WRF data
    approach="Time",  
    
    ## Heat Index specific arguments
    variable="NOAA Heat Index", 
    metric_calc="max", # daily maximum
    heat_idx_threshold=90, # Heat Index Threshold
    units="degF", # change units
    
    ## Export
    export_method="raw",
    file_format="csv",
    batch_mode=False,
)

Calculating Days per year with Max Daily NOAA Heat Index above a heat index threshold of 90 degF for all seasons from 2030_to_2060

--- Selecting Data Points --- 

Selecting data for (35.43424, -119.05524)


/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:1227: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()


Filtering WRF data for bias-adjusted simulations.

--- Loading Data into Memory ---

Point (35.445045471191406, -119.06060791015625)
Processing data to read 5.18 MB of data into memory... 
[########################################] | 100% Completed | 104.57 s
Complete!

--- Exporting Raw Data ---
Exporting specified data to CSV...
NOTE: File metadata will be written in /home/jovyan/cae-notebooks/heat_index_raw_data_35-445045N_119-060608W_metadata.txt. We recommend you download this along with the CSV for your records.
Saved! You can find your file(s) in the panel to the left and download to your local machine from there.


#### Example: Global Warming Level approach with WRF
Example scenario: I want to calculate "likely summer day high temperature with a 2°C warming level, where likely is the 50th percentile, in Celsius, using all available WRF data (bias-adjusted and non-bias-adjusted), and export both the raw and calculated metric data". I would input:

In [7]:
data = cava_data(
    ## Set-up
    example_locs.iloc[:1], # select a single location
    downscaling_method="Dynamical",  # WRF data
    approach="Warming Level",
    warming_level=2.0,
    wrf_bias_adjust=False, # return all WRF models
    
    ## Likely seasonal event specific arguments
    variable="Air Temperature at 2m", 
    metric_calc="max", # daily high temperature
    percentile=50, # likeliness percentile
    season="summer", # season
    units="degC", # change units
    
    ## Export
    export_method="both",
    file_format="NetCDF",
)

Calculating 50th percentile of Daily Max of Air Temperature at 2m for summer seasons for Warming Level 2.0°C

--- Selecting Data Points --- 

Selecting data for (35.43424, -119.05524)


/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:1227: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()



--- Loading Data into Memory ---

Point (35.445045471191406, -119.06060791015625)
Processing data to read 2.02 MB of data into memory... 
[########################################] | 100% Completed | 57.53 s
Complete!

--- Exporting Raw Data ---
Exporting specified data to NetCDF...
Saving file locally as NetCDF4...
[########################################] | 100% Completed | 101.15 ms
Saved! You can find your file in the panel to the left and download to your local machine from there.

--- Calculating Metrics ---
Calculating...


Complete!

--- Exporting Metric Data ---
Exporting specified data to NetCDF...
Saving file locally as NetCDF4...
[########################################] | 100% Completed | 101.01 ms
Saved! You can find your file in the panel to the left and download to your local machine from there.


#### Example: Global Warming Level approach with LOCA2
Example scenario: I want to calculate "likely winter day high temperature with a 2°C warming level, where likely is the 60th percentile, in Celsius, using LOCA2 data, and export both the raw and calculated metric data". I would input: 

In [8]:
data = cava_data(
    ## Set-up
    example_locs.iloc[:1], # select a single location
    downscaling_method="Statistical",  # LOCA2 data
    approach="Warming Level",
    warming_level=2.0,
    
    ## Likely seasonal event specific arguments
    variable="Air Temperature at 2m", 
    metric_calc="max", # daily high temperature
    percentile=60, # likeliness percentile
    season="winter", # season
    units="degC", # change units
    
    ## Export
    export_method="both",
    file_format="NetCDF",
)

Calculating 60th percentile of Daily Max of Maximum air temperature at 2m for winter seasons for Warming Level 2.0°C

--- Selecting Data Points --- 

Selecting data for (35.43424, -119.05524)


/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:1227: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()
/home/jovyan/.local/lib/python3.12/site-packages/climakitae/core/data_load.py:638: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'member_id' ('member_id',) The recommendation is to set join explicitly for this case.
  all_ssps = xr.concat(



--- Loading Data into Memory ---

Point (35.421875, -119.046875)
Processing data to read 1.33 MB of data into memory... 
[########################################] | 100% Completed | 349.28 s
Complete!

--- Exporting Raw Data ---

Exporting Statistical data in most granular time availability (daily)

Exporting specified data to NetCDF...
Saving file locally as NetCDF4...
[########################################] | 100% Completed | 101.11 ms
Saved! You can find your file in the panel to the left and download to your local machine from there.

--- Calculating Metrics ---
Calculating...


Complete!

--- Exporting Metric Data ---
Exporting specified data to NetCDF...
Saving file locally as NetCDF4...
[########################################] | 100% Completed | 101.25 ms
Saved! You can find your file in the panel to the left and download to your local machine from there.


#### Example: Heat Index, in batch mode for multiple locations
Example scenario: I want to calculate "the number of days per year that the Heat Index exceeds 104°F between 1990-2010 in the Historical Reconstruction data for many locations, and export only the calculated metric data". I would input:

In [9]:
data = cava_data(
    ## Set-up
    example_locs, # no subsetting for a single location from input list
    time_start_year=1990,
    time_end_year=2010,
    historical_data="Historical Reconstruction", # selecting reconstruction data
    approach="Time",  
    
    ## Heat Index specific arguments
    variable="NOAA Heat Index", 
    metric_calc="max", # daily maximum
    heat_idx_threshold=104, # Heat Index Threshold
    units="degF", # change units
    
    ## Export
    export_method="calculate",
    file_format="NetCDF",
    batch_mode=True, # batch mode - optimized for multiple locations
)

Calculating Days per year with Max Daily NOAA Heat Index above a heat index threshold of 104 degF for all seasons from 1990_to_2010

--- Selecting Data Points --- 

[WARNING] Batch mode is active, 'separate_files' is being set to False. 
[WARNING] This means that your output will be a single file with all locations included. 

Batch retrieving all 33 points passed in...



/home/jovyan/.local/lib/python3.12/site-packages/climakitae/tools/batch.py:42: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!! Returned data array is huge. Operations could take 10x to infinity longer than 1GB of data !!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Processing 33 coordinate pair(s)...
  Spatial dimensions: y, x
  Data grid size: 472 x 223
  Found 33 valid point(s) within data extent
  Checking landmask for valid land points...
  31/33 points are on land
  2 point(s) need neighbor search...
  Fixed 2/2 points with valid neighbors
  Extracting gridcells...
Done! Extracted 33 gridcell(s)

--- Loading Data into Memory ---

Processing data to read 23.16 MB of data into memory... 
[########################################] | 100% Completed | 435.63 s
Complete!

--- Calculating Metrics ---
Calculating...


Complete!

--- Exporting Metric Data ---
Exporting specified data to NetCDF...
Saving file locally as NetCDF4...
[###########################

#### Example: Likely seasonal event, in batch mode for multiple locations with LOCA2
Example scenario: I want to calculate "likely summer day high temperature for 1.5°C warming level, where likely is the 70th percentile, in Celsius, using all available LOCA2 data, and export only the calculated metric data" for **multiple locations**. I would input:

**Note:** Batch mode for LOCA2 data using the warming levels approach resets to `batch_mode = False` regardless of your setting here due to optimization constraints, but will compute all desired metrics. This will take quite some time to run (**2 locations** with warming levels with LOCA2 data takes **approx. 1 hour to run**) -- hang tight! Improvements in this space is forthcoming!

In [ ]:
data = cava_data(
    ## Set-up
    example_locs, # select multiple locations
    downscaling_method="Statistical",  # LOCA data 
    approach="Warming Level",  
    warming_level=1.5, 
    
    ## Likely seasonal event specific arguments
    variable="Air Temperature at 2m",
    metric_calc="max", # daily maximum temperature
    season='summer', # change season
    percentile=70, # change percentile
    units="degC", # change units

    ## Export
    export_method="calculate",
    file_format="NetCDF",
    batch_mode=False  # batch mode - optimized for multiple locations
)

Calculating 70th percentile of Daily Max of Maximum air temperature at 2m for summer seasons for Warming Level 1.5°C

--- Selecting Data Points --- 

Selecting data for (35.43424, -119.05524)


/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:1227: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()
/home/jovyan/.local/lib/python3.12/site-packages/climakitae/core/data_load.py:638: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'member_id' ('member_id',) The recommendation is to set join explicitly for this case.
  all_ssps = xr.concat(


Selecting data for (33.61876, -114.71451)


/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:1227: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()
/home/jovyan/.local/lib/python3.12/site-packages/climakitae/core/data_load.py:638: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'member_id' ('member_id',) The recommendation is to set join explicitly for this case.
  all_ssps = xr.concat(


Selecting data for (34.19966, -118.36543)


...

Selecting data for (34.067, -117.65)


/home/jovyan/.local/lib/python3.12/site-packages/climakitae/explore/vulnerability.py:1227: DeprecationWarning: DataParameters.retrieve() is deprecated and will be removed in climakitae 2.0.0 (targeting January 2027). Use climakitae.new_core.user_interface.ClimateData instead.
  data = selections.retrieve()
/home/jovyan/.local/lib/python3.12/site-packages/climakitae/core/data_load.py:638: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'member_id' ('member_id',) The recommendation is to set join explicitly for this case.
  all_ssps = xr.concat(



--- Loading Data into Memory ---

Point (35.421875, -119.046875)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 301.26 s
Complete!
Point (33.609375, -114.703125)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 227.81 s
Complete!
Point (34.203125, -118.359375)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 340.37 s
Complete!
Point (34.765625, -114.609375)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 268.22 s
Complete!
Point (36.765625, -119.734375)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 288.20 s
Complete!
Point (32.828125, -115.578125)
Processing data to read 1.36 MB of data into memory... 
[#####################################

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[########################################] | 100% Completed | 283.28 s
Complete!
Point (32.734375, -117.171875)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 140.41 s
Complete!
Point (34.421875, -119.828125)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 161.26 s
Complete!
Point (35.234375, -120.640625)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 166.37 s
Complete!
Point (32.828125, -116.984375)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 140.41 s
Complete!
Point (37.609375, -122.359375)
Processing data to read 1.36 MB of data into memory... 
[########################################] | 100% Completed | 259.28 s
Complete!
Point (37.359375, -121.921875)
Processing data to read 1.36 MB of data into mem

#### Example: 1-in-X precipitation event
Example scenario: I want to calculate "1-in-10 year precipitation event using the GEV distribution, in inches, for 2070-2090 with SSP 3-7.0, using LOCA2 data, and export both the raw and calculated metric data." I would input:

**Notes**: 
- For daily precipitation 1-in-X events (i.e., 24-hour), we recommend the use of LOCA2 data instead of WRF data. If looking for a non-24-hour event, WRF data must be used, as the LOCA2 data is not available at hourly time steps. 
- The goodness of fit on the distribution is provided for 1-in-X precipitation events. The p-value of the distribution fit to the data is provided during the calculation step and as an attribute in the final data object. For 1-in-X precipitation events, **we recommend the use of "gev" as the distribution** for the first distribution test. GEV allows for a continuous range of different shapes, and will reduce to either Gumbel, Weibull, or Generalized Pareto distributions under different conditions. GEV is typically a better fit than the 3 individaul distributions, and is a common distribution in hydrological applications. If the **p-value is less than 0.05**, this indicates that the **selected distribution is not a good fit for the data**, and we recommend choosing a different distribution and re-running the `cava_data` function. 
- In certain geographic regions, the selection of a high return period event (e.g., 1-in-1000) may produce unrealistically high precipitation values. This is primarily a limitation of the data sample size beyond what the data can reasonably estimate, where a small change in noise will produce a large change in the distribution tails. 

In [ ]:
data = cava_data(
    ## Set-up
    example_locs.iloc[:1], # select a single location
    time_start_year=2070,
    time_end_year=2090,
    downscaling_method="Statistical",  # LOCA2 data 
    approach="Time",  
    ssp_data=["SSP 3-7.0"], # ssp selection
    
    ## 1-in-X event specific arguments
    variable="Precipitation (total)",
    metric_calc="max", # daily maximum precipitation
    one_in_x=10, # One-in-X
    distr="gev", # change distribution
    units="inches", # change units
    
    ## Export
    export_method="both",
    file_format="NetCDF",
)

#### Example: 1-in-X precipitation event with `duration`, `groupby`, and `grouped_duration`
Example scenario: I want to calculate a 1-in-100 year precipitation event using the GEV distribution, in inches, for 2070-2090, using only the bias-adjusted WRF data. The three event-window parameters can be combined to define the event:
- `duration=(3, 'hour')` — sub-daily event window (hourly WRF only; only 'hour' units supported)
- `groupby=(1, 'day')` — groups sub-daily values into daily blocks before computing block maxima (only 'day' units supported)
- `grouped_duration=(5, 'day')` — multi-day event window after grouping; requires `groupby` to be set (only 'day' units supported)

The example below uses `duration` and `groupby` together (daily max of 3-hour windows). To instead compute a 5-day event, comment out `duration` and uncomment `grouped_duration`.

For more details on how to use these parameters, see the <a href="../../../analysis/threshold_event_types.ipynb" target="_blank">Threshold Tools notebook</a>.

**Note**: For daily precipitation 1-in-X events (i.e., 24-hour), we recommend LOCA2 data. For sub-daily events, WRF hourly data must be used.

In [ ]:
data = cava_data(
    ## Set-up
    example_locs.iloc[:1], # select a single location
    downscaling_method="Dynamical",  # WRF data (required for sub-daily hourly data)
    approach="Time",
    time_start_year=2070,
    time_end_year=2090,
    wrf_bias_adjust=True, # return bias-adjusted WRF models
    
    ## 1-in-X event specific arguments
    variable="Precipitation (total)",
    metric_calc="max",
    one_in_x=100,
    distr="gev",
    duration=(3, 'hour'),   # sub-daily event window
    groupby=(1, 'day'),     # take daily max of duration windows
    # grouped_duration=(5, 'day'),  # alternative: 5-consecutive-day event (comment out duration)
    units="inches",
    
    ## Export
    export_method="both",
    file_format="NetCDF",
)

#### Example: Looking at the `cava_data` output
It may be useful to look at the `cava_data` output within this notebook to assess the results and make any changes to the data request. After running the `cava_data` function, in a new cell you can type `data` to view the xarray data object. Depending on your export setting ("raw", "calculate", "both"), you can also view the data object in a more user-friendly xarray view. We provide the code to do so in the next cell -- select which option matches your `cava_data` run and the export option you would like to view! 

In [ ]:
data # looking at the full xarray data object; will be a dictionary of data arrays!
# data['calc_data'] # looking at just the calculated data metric
# data['raw'] # looking at just the raw input data